# Exploring a global MPAS mesh with pyramids-mesh

**MPAS** — the *Model for Prediction Across Scales* — is the unstructured global modelling framework behind a
family of NCAR/DOE atmosphere and ocean models. Its horizontal grid is a **Spherical Centroidal Voronoi
Tessellation**: a mostly-hexagonal mesh of Voronoi cells tiling the sphere, with exactly twelve pentagons where
the icosahedral seed points sit (the same reason a football carries twelve pentagons). Coordinates are stored in
**radians** of longitude/latitude, and there is no map projection.

The file we open is a genuine production grid: the **`x1.1024002`** quasi-uniform global mesh — roughly **one
million cells** at a near-constant cell spacing over the whole planet, the kind of grid a global atmosphere run
is actually integrated on. At that size the mesh is far too large to draw whole, so this notebook leans on
pyramids' `crop(...)` to carve out a small window and make the individual Voronoi cells visible. The `mpas`
driver reads the file into a pyramids `MeshDataset`, handling the 1-based connectivity and fill values for you.

## 1. Imports

`MeshDataset` is the object the `mpas` driver hands back (a `Mesh2d`/`Mesh1d` wrapper). Importing from
`pyramids` also puts its vendored GDAL/PROJ data in place, so nothing here is order-sensitive in a notebook —
just import the names you use. `add_tiles` would add a web basemap *when a mesh is georeferenced*; an MPAS grid
never is, so we only ever call it behind an `if ds.epsg` guard.

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from pyramids.netcdf.ugrid import MeshDataset
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the grid

This notebook lives in `docs/examples/real-data/`, so the shared `examples/data/real-data/` corpus is three
levels up. The MPAS grid lives under `mpas/`. Paths inside a notebook are written **relative to the notebook
itself**, and pyramids' `read_file` takes a `pathlib.Path` directly — we pass the `Path`, never `str(path)`.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
grid = DATA / "mpas" / "x1.1024002.grid.nc"
grid, grid.exists()

## 3. Read the mesh

`MeshDataset.read_file(path, driver="mpas")` dispatches the file to the `mpas` driver, which parses the Voronoi
topology and hands back a `MeshDataset` — the same pyramids type every other driver produces. We name the driver
explicitly because several formats share the `.nc` suffix. The file is ~1.4 GB on disk, so we time the read to
see how a million-cell global grid loads.

You may see a winding-normalisation warning (a few clockwise faces re-wound to counter-clockwise) — that is
pyramids tidying the face orientation, informational rather than an error.

In [ ]:
t0 = time.perf_counter()
ds = MeshDataset.read_file(grid, driver="mpas")
read_seconds = time.perf_counter() - t0
print(f"read in {read_seconds:.2f} s")
ds

A ~1.4 GB global grid reads into a 2-D `MeshDataset` in a few seconds: **1 024 002 faces** (the MPAS *cells*)
built from **2 048 000 nodes** (the Voronoi *vertices*). This is a grid-only file — no prognostic fields are
attached — so there are no data variables yet.

## 4. Choose the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Immediately after that first
read we call cleopatra's own chooser once: it detects the notebook and selects the inline backend (and falls
back to `Agg` in a plain script or in CI), so every figure below embeds natively with no manual rasterising.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 5. Explore the dataset

A `MeshDataset` exposes summary properties describing the whole mesh at a glance: `n_node` / `n_face` (size),
`crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent), and
`data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 4) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

**One million cells** on **two million vertices**, `crs`/`epsg` both `None`, and no data variables — a bare
global grid. The `bounds` look small for a whole planet (longitude ≈ `-3.14 → 3.14`, latitude ≈ `-1.57 → 1.57`)
because they are in **radians**, not degrees — the next section makes that explicit.

## 6. Coordinates are in RADIANS

MPAS stores node longitude/latitude in **radians**. The tell is the range: longitude spans `-π → π ≈ ±3.1416`
and latitude `-π/2 → π/2 ≈ ±1.5708`. We print the extents against those constants, then convert to the familiar
degrees (`× 180/π`) so the global coverage is obvious.

In [ ]:
nx, ny = ds.mesh.node_x, ds.mesh.node_y
pd.DataFrame({
    "radians": [float(nx.min()), float(nx.max()), float(ny.min()), float(ny.max())],
    "degrees": [np.degrees(v) for v in (nx.min(), nx.max(), ny.min(), ny.max())],
}, index=["lon min", "lon max", "lat min", "lat max"])

Converted to degrees the picture is unmistakable: longitude runs the full **−180° → 180°** and latitude the
full **≈ −90° → 90°** — a genuinely global mesh. Any CRS-aware operation would first convert these radians to
degrees and assign a geographic CRS; the raw file leaves `crs` unset.

## 7. Voronoi cells and a variable vertex count

MPAS cells are Voronoi polygons — mostly hexagons, with twelve pentagons at the icosahedral seeds. Because the
cells do not all have the same number of corners, the `face_node_connectivity` table is **padded**: it is as
wide as the largest cell, and shorter cells fill their unused slots with the fill value (`-1`). Counting the
valid (non-`-1`) entries per row recovers the actual vertex count of each cell.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
widths = (conn >= 0).sum(axis=1)
mix = (
    pd.Series(widths)
    .value_counts()
    .rename_axis("nodes_per_face")
    .rename({5: "5 (pentagon)", 6: "6 (hexagon)"})
    .to_frame("n_faces")
)
print("connectivity shape:", conn.shape)
mix

The connectivity table is padded well beyond the cells that use it, yet the real vertex counts are only two
values: **≈ 1 023 990 hexagons** and exactly **12 pentagons** — the structural signature of a spherical Voronoi
mesh seeded from an icosahedron. A triangular grid would instead show a single `k = 3`; here the variable count
is why the connectivity has to be padded at all.

## 8. The global mesh is too large to draw

At one million faces, drawing the full wireframe (or a global data map) would mean rendering a million polygons
— slow to build, and a solid black smear on screen at any sensible figure size, with the individual cells far
too small to see. So we do **not** plot the whole mesh. Instead we use pyramids' `crop(...)` to carve out a
small lon/lat window of a few thousand cells, where the Voronoi hexagons are large enough to make out.

## 9. Crop to a small window

`crop(bbox=(xmin, ymin, xmax, ymax))` returns a **new** `MeshDataset` holding only the faces whose centroids
fall inside the box — the mesh equivalent of a spatial window, and the original `ds` is left untouched. The box
is given in the mesh's own units, so **in radians**. We take a `±0.1 rad` window centred on the origin
(longitude 0°, the equator) — about **±5.7°** — which should keep a few thousand cells. We print `n_face` to
confirm the size before drawing anything.

In [ ]:
half = 0.1  # radians (~5.7 degrees) half-width
bbox = (-half, -half, half, half)
cropped = ds.crop(bbox=bbox)
pd.Series({
    "bbox_radians": bbox,
    "bbox_degrees": tuple(round(np.degrees(v), 2) for v in bbox),
    "n_face_global": ds.n_face,
    "n_face_cropped": cropped.n_face,
    "n_node_cropped": cropped.n_node,
})

The crop drops the mesh from **over a million faces to a few thousand** — a patch small enough to render
cell-by-cell while still being a faithful slice of the real global grid.

With the window small enough, `plot_outline()` draws the cell edges of the cropped subset through cleopatra.
The tile call is guarded by `if ds.epsg`, so it is skipped for this un-georeferenced grid.

In [ ]:
_g = cropped.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)
_g.ax.set_title("MPAS x1.1024002 — cropped window")
_g.ax.set_aspect("equal")

Now the Voronoi tessellation is visible: a near-regular tiling of **hexagonal cells** of almost uniform size —
exactly what *quasi-uniform* means. Near the equator the lon/lat flattening barely distorts them, so the
honeycomb pattern comes through cleanly.

## 10. A per-cell field on the subset

This grid-only file carries no field values, so we synthesise one that is easy to interpret: each cell's
**latitude in degrees**, computed from its centroid. We attach it as a face-located variable with
`with_variable(...)`, which returns a **new** dataset, then colour the mesh by it — the generic pyramids
data-map that any real per-cell field (temperature, pressure, …) would use.

In [ ]:
cx, cy = cropped.mesh.face_centroids
cell_lat_deg = np.degrees(np.asarray(cy))
d2 = cropped.with_variable("cell_lat_deg", cell_lat_deg, location="face")
_g = d2.plot("cell_lat_deg", cmap="viridis", colorbar=True, title="Cell latitude (degrees)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)
_g.ax.set_aspect("equal")

Each hexagon is shaded by its latitude, producing a smooth south-to-north gradient across the window — the
cell shapes and the colour field together. The exact same call draws any physical field a solution file would
carry; the only thing synthetic here is the values.

`summary()` reports per-variable statistics (count, min, max, mean, std) for every attached field — the
numeric companion to the colour map.

In [ ]:
d2.summary()

A single row for `cell_lat_deg`: its `count` equals the cropped face count, and the min/max span the ≈ ±5.7°
window we cropped to — the map's gradient, confirmed numerically.

## 11. GIS operations, for free

Because the cropped mesh is still a pyramids object, the pyramids GIS layer applies directly — none of these
methods live in `pyramids-mesh`, which only reads the file. We tour three on the subset (running them on the
million-cell global mesh would be needlessly heavy, and the periodic longitude seam complicates a global
dissolve).

### Faces as a `GeoDataFrame`

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile/GeoPackage or for spatial joins. Each Voronoi cell becomes an ordinary
shapely polygon.

In [ ]:
gdf = cropped.to_geodataframe()
print(type(gdf).__name__, gdf.shape)
gdf.head()

One polygon per face, one row per cell. The CRS is unset — inherited from the mesh's `None` — so you would
assign a geographic CRS (after the radian→degree conversion) before exporting to a shapefile.

### The outer boundary

`bounding_polygon()` dissolves all the cropped faces into the single enclosing polygon — the outline of the
window we cut. (On the full global grid this dissolve hits the periodic longitude seam and is best skipped;
on a small interior patch it is well-behaved.)

In [ ]:
poly = cropped.bounding_polygon()
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="red", linewidth=2)
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
ax.set_aspect("equal")
ax.set_title("bounding polygon of the cropped window")
pd.Series({"geom_type": poly.geom_type, "area_rad2": round(poly.area, 4)})

A single `Polygon` enclosing the cropped patch — an ordinary shapely geometry, the gateway to the
shapely/geopandas ecosystem for clipping and spatial joins. Its area (in radian² units) matches the small
window we cut.

### Connected components

`connected_components()` labels each face by which edge-connected piece of the mesh it belongs to, and
`sparse_adjacency()` returns the face-to-face adjacency as a SciPy sparse matrix — the graph a solver walks
between neighbouring cells.

In [ ]:
adj = cropped.sparse_adjacency()
labels = cropped.connected_components()
counts = np.bincount(labels)
pd.Series({
    "adjacency_shape": adj.shape,
    "adjacency_nnz": int(adj.nnz),
    "n_components": int(np.unique(labels).size),
    "largest_component": int(counts.max()),
})

Almost every cell falls in **one** dominant component spanning the window; any extra labels are a handful of
single cells left edge-isolated by the crop (their centroid landed inside the box while their edge-neighbours'
centroids fell just outside). On the uncropped sphere the whole mesh is of course one contiguous surface.

## Takeaway

- The `mpas` driver reads a real **one-million-cell** quasi-uniform global MPAS grid (~1.4 GB) into a pyramids
  `MeshDataset` in **seconds**, handling the 1-based connectivity and fill values.
- MPAS cells are **Voronoi polygons** — ≈ 1 023 990 hexagons plus exactly **12 pentagons** — so the
  `face_node_connectivity` has a **variable, padded** vertex count rather than a single fixed width.
- Coordinates are in **radians** (longitude ±π, latitude ±π/2); converted to degrees they cover the whole
  globe (±180°, ±90°). `crs`/`epsg` are `None` and the file carries no fields.
- A million faces are **too large to draw**, but `crop(bbox=...)` with a radian window makes any region
  explorable — the hexagonal Voronoi cells become visible, and a per-cell field maps cleanly.
- The full pyramids GIS layer (`to_geodataframe`, `bounding_polygon`, `connected_components`,
  `sparse_adjacency`) applies to the mesh for free.